## Transformer-based Slot Filling Model

### 1. Imports and Initial Setup

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

train_url = "/content/drive/MyDrive/DL LAB S2/7/Dataset/atis.train.csv"
test_url = "/content/drive/MyDrive/DL LAB S2/7/Dataset/atis.test.csv"

### 2. Load Data


In [2]:
df_train = pd.read_csv(train_url, index_col=0)
df_test = pd.read_csv(test_url, index_col=0)

pd.set_option('display.max_colwidth', None)

df_small = pd.DataFrame(columns=['tokens','slots'])
j = 0
for i in df_train.intent.unique():
    df_small.loc[j] = df_train[df_train.intent==i].iloc[0]
    j += 1

print("df_train head:")
display(df_train.head())
print("\ndf_test head:")
display(df_test.head())
print("\ndf_small head:")
display(df_small.head())

df_train head:


,tokens,slots,intent
id,,,
train-00001,BOS what is the cost of a round trip flight from pittsburgh to atlanta beginning on april twenty fifth and returning on may sixth EOS,O O O O O O O B-round_trip I-round_trip O O B-fromloc.city_name O B-toloc.city_name O O B-depart_date.month_name B-depart_date.day_number I-depart_date.day_number O O O B-return_date.month_name B-return_date.day_number O,atis_airfare
train-00002,BOS now i need a flight leaving fort worth and arriving in denver no later than 2 pm next monday EOS,O O O O O O O B-fromloc.city_name I-fromloc.city_name O O O B-toloc.city_name B-arrive_time.time_relative I-arrive_time.time_relative I-arrive_time.time_relative B-arrive_time.time I-arrive_time.time B-arrive_date.date_relative B-arrive_date.day_name O,atis_flight
train-00003,BOS i need to fly from kansas city to chicago leaving next wednesday and returning the following day EOS,O O O O O O B-fromloc.city_name I-fromloc.city_name O B-toloc.city_name O B-depart_date.date_relative B-depart_date.day_name O O B-return_date.date_relative I-return_date.date_relative I-return_date.date_relative O,atis_flight
train-00004,BOS what is the meaning of meal code s EOS,O O O O O O B-meal_code I-meal_code I-meal_code O,atis_abbreviation
train-00005,BOS show me all flights from denver to pittsburgh which serve a meal for the day after tomorrow EOS,O O O O O O B-fromloc.city_name O B-toloc.city_name O O O B-meal O B-depart_date.today_relative I-depart_date.today_relative I-depart_date.today_relative I-depart_date.today_relative O,atis_flight



df_test head:


,tokens,slots,intent
id,,,
test-00001,BOS what are the coach flights between dallas and baltimore leaving august tenth and returning august twelve EOS,O O O O B-class_type O O B-fromloc.city_name O B-toloc.city_name O B-depart_date.month_name B-depart_date.day_number O O B-return_date.month_name B-return_date.day_number O,atis_flight
test-00002,BOS i want a flight from nashville to seattle that arrives no later than 3 pm EOS,O O O O O O B-fromloc.city_name O B-toloc.city_name O O B-arrive_time.time_relative I-arrive_time.time_relative I-arrive_time.time_relative B-arrive_time.time I-arrive_time.time O,atis_flight
test-00003,BOS i need a flight leaving kansas city to chicago leaving next wednesday and returning the following day EOS,O O O O O O B-fromloc.city_name I-fromloc.city_name O B-toloc.city_name O B-depart_date.date_relative B-depart_date.day_name O O B-return_date.date_relative I-return_date.date_relative I-return_date.date_relative O,atis_flight
test-00004,BOS explain meal codes sd d EOS,O O B-meal O B-meal_code I-meal_code O,atis_abbreviation
test-00005,BOS show me all flights from atlanta to san francisco which leave the day after tomorrow after 5 o'clock pm EOS,O O O O O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name O O B-depart_date.today_relative I-depart_date.today_relative I-depart_date.today_relative I-depart_date.today_relative B-depart_time.time_relative B-depart_time.time I-depart_time.time I-depart_time.time O,atis_flight



df_small head:


,tokens,slots
0,BOS what is the cost of a round trip flight from pittsburgh to atlanta beginning on april twenty fifth and returning on may sixth EOS,O O O O O O O B-round_trip I-round_trip O O B-fromloc.city_name O B-toloc.city_name O O B-depart_date.month_name B-depart_date.day_number I-depart_date.day_number O O O B-return_date.month_name B-return_date.day_number O
1,BOS now i need a flight leaving fort worth and arriving in denver no later than 2 pm next monday EOS,O O O O O O O B-fromloc.city_name I-fromloc.city_name O O O B-toloc.city_name B-arrive_time.time_relative I-arrive_time.time_relative I-arrive_time.time_relative B-arrive_time.time I-arrive_time.time B-arrive_date.date_relative B-arrive_date.day_name O
2,BOS what is the meaning of meal code s EOS,O O O O O O B-meal_code I-meal_code I-meal_code O
3,BOS now show me ground transportation in houston on monday afternoon EOS,O O O O O O O B-city_name O B-day_name B-period_of_day O
4,BOS what are the restrictions on the cheapest one way fare between boston and oakland EOS,O O O O O O O B-cost_relative B-round_trip I-round_trip O O B-fromloc.city_name O B-toloc.city_name O


### 3. Prepare Data for Vectorization



In [3]:
query_data_train = df_train['tokens'].values
slot_data_train = df_train['slots'].values

query_data_test = df_test['tokens'].values
slot_data_test = df_test['slots'].values

max_query_length = 30

print(f"Max query length set to: {max_query_length}")

Max query length set to: 30


### 4. Text Vectorization for Queries



In [4]:
text_vectorization_query = keras.layers.TextVectorization(
    output_sequence_length=max_query_length
)

text_vectorization_query.adapt(query_data_train)
query_vocab_size = text_vectorization_query.vocabulary_size()

source_train = text_vectorization_query(query_data_train)
source_test = text_vectorization_query(query_data_test)

print(f"Query vocabulary size: {query_vocab_size}")
print(f"Example of vectorized training query (first sample):\n{source_train[0].numpy()}")

Query vocabulary size: 874
Example of vectorized training query (first sample):
[  3  11  22   7 161  34  15  57  51   9   5  24   4  21 783   8 188 101
 236  19 355   8 171 255   2   0   0   0   0   0]


### 5. Text Vectorization for Slots



In [5]:
text_vectorization_slots = keras.layers.TextVectorization(
    output_sequence_length=max_query_length,
    standardize=None # Do not standardize slot labels
)

text_vectorization_slots.adapt(slot_data_train)
slot_vocab_size = text_vectorization_slots.vocabulary_size()

target_train = text_vectorization_slots(slot_data_train)
target_test = text_vectorization_slots(slot_data_test)

print(f"Slot vocabulary size: {slot_vocab_size}")
print(f"Example of vectorized training slot sequence (first sample):\n{target_train[0].numpy()}")

Slot vocabulary size: 103
Example of vectorized training slot sequence (first sample):
[  2   2   2   2   2   2   2  14  15   2   2   4   2   3   2   2  12  11
  29   2   2   2 101 102   2   0   0   0   0   0]


### 6. Define Transformer Encoder Layer



In [6]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads):
        super().__init__()
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim)
            ]
        )
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

### 7. Define Token and Position Embedding Layer



In [7]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
        self.pos_emb = layers.Embedding(
            input_dim=maxlen,
            output_dim=embed_dim
        )

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

### 8. Build and Compile the Transformer Model


In [8]:
embed_dim = 512
dense_dim = 64
num_heads = 5

embedding = TokenAndPositionEmbedding(
    max_query_length,
    query_vocab_size,
    embed_dim
)

te = TransformerEncoder(embed_dim, dense_dim, num_heads)

inputs = keras.layers.Input(shape=(max_query_length,))
x = embedding(inputs)
encoder_out = te(x)

x = keras.layers.Dense(128, activation='relu')(encoder_out)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(slot_vocab_size, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 30)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 30, 512)        │       462,848 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, 30, 512)        │     1,114,682 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 30, 128)        │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 30, 103)        │        13,287 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,656,481 (6.32 MB)

 Trainable params: 1,656,481 (6.32 MB)

 Non-trainable params: 0 (0.00 B)

### 9. Train the Model



In [9]:
BATCH_SIZE = 64
epochs = 10

history = model.fit(
    source_train,
    target_train,
    batch_size=BATCH_SIZE,
    epochs=epochs,
    validation_split=0.2 # Added for better monitoring of overfitting
)

Epoch 1/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 26s 436ms/step - accuracy: 0.8883 - loss: 0.5580 - val_accuracy: 0.9758 - val_loss: 0.0610
Epoch 2/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 23s 429ms/step - accuracy: 0.9491 - loss: 0.1816 - val_accuracy: 0.9782 - val_loss: 0.0522
Epoch 3/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 23s 430ms/step - accuracy: 0.9558 - loss: 0.1390 - val_accuracy: 0.9789 - val_loss: 0.0493
Epoch 4/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 41s 422ms/step - accuracy: 0.9605 - loss: 0.1171 - val_accuracy: 0.9812 - val_loss: 0.0475
Epoch 5/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 41s 436ms/step - accuracy: 0.9637 - loss: 0.1050 - val_accuracy: 0.9829 - val_loss: 0.0439
Epoch 6/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 41s 426ms/step - accuracy: 0.9679 - loss: 0.0920 - val_accuracy: 0.9846 - val_loss: 0.0407
Epoch 7/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 41s 437ms/step - accuracy: 0.9745 - loss: 0.0776 - val_accuracy: 0.9912 - val_loss: 0.0276
Epoch 8/10
54/54 ━━━━━━━━━━━━━━━━━━━━ 23s 431ms/step - accuracy: 0.9799 - loss: 0.0646 - val_accu

### 10. Evaluate the Model


In [10]:
print("Model Evaluation on Test Data:")
model.evaluate(source_test, target_test)

Model Evaluation on Test Data:
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - accuracy: 0.9841 - loss: 0.0625


[0.06253430992364883, 0.9841298460960388]

### 11. Define Slot Filling Accuracy Function



In [11]:
def slot_filling_accuracy(actual, predicted, only_slots=False):
    not_padding = np.not_equal(actual, 0)

    if only_slots:
        # Get the index of the 'O' token (non-slot token)
        non_slot_token_index = text_vectorization_slots(['O']).numpy()[0, 0]
        # Identify positions that are actual slots (not padding AND not 'O')
        slots = np.not_equal(actual, non_slot_token_index)
        # Calculate correct predictions only for actual slots
        correct_predictions = np.equal(actual, predicted)[not_padding & slots]
    else:
        # Calculate correct predictions for all non-padding tokens
        correct_predictions = np.equal(actual, predicted)[not_padding]

    return np.mean(correct_predictions)

### 12. Calculate and Print Custom Accuracies



In [12]:
predicted_raw = model.predict(source_test)
predicted = np.argmax(predicted_raw, axis=-1).reshape(-1)
actual = np.array(target_test).reshape(-1)

acc = slot_filling_accuracy(actual, predicted, only_slots=False)
acc_slots = slot_filling_accuracy(actual, predicted, only_slots=True)

print(f'Overall Accuracy = {acc:.3f}')
print(f'Accuracy on Slots (excluding padding and "O") = {acc_slots:.3f}')

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step
Overall Accuracy = 0.964
Accuracy on Slots (excluding padding and "O") = 0.895


### 13. Define Prediction Function



In [13]:
def predict_slots_query(query):
    # Vectorize the input query
    sentence = text_vectorization_query([query])
    # Get raw model prediction and take argmax to get predicted token indices
    prediction = np.argmax(model.predict(sentence, verbose=0), axis=-1)[0]

    # Create inverse vocabulary to decode numerical predictions back to words
    inverse_vocab = dict(enumerate(text_vectorization_slots.get_vocabulary()))
    # Decode the predicted sequence of indices into slot labels
    decoded_prediction = " ".join(inverse_vocab[int(i)] for i in prediction)
    return decoded_prediction

### 14. Demonstrate Prediction


In [14]:
example_query = "cheapest flight to fly from MIT to Mars"
predicted_slots = predict_slots_query(example_query)
print(f"Query: {example_query}")
print(f"Predicted Slots: {predicted_slots}")

Query: cheapest flight to fly from MIT to Mars
Predicted Slots: B-cost_relative O O O O O O B-toloc.city_name                      


### 15. Predict with another example

In [15]:
example_query_2 = "show me flights from Boston to New York on Tuesday morning"
predicted_slots_2 = predict_slots_query(example_query_2)
print(f"Query: {example_query_2}")
print(f"Predicted Slots: {predicted_slots_2}")

Query: show me flights from Boston to New York on Tuesday morning
Predicted Slots: O O O O B-fromloc.city_name O B-toloc.city_name I-toloc.city_name O B-depart_date.day_name B-depart_time.period_of_day                   
